# Step 2 — 00. Checkpoint registration

목적은 공개 checkpoint의 출처, 로컬 SHA-256, 전처리, backbone,
학습 데이터 표기, 512D 출력과 Grad-CAM target layer를 하나의
불변 `ModelSpec`으로 등록하는 것입니다.

이 노트북은 모델을 학습하지 않습니다. 아래 값을 논문/공식 저장소와
실제 파일을 확인하여 직접 채우기 전에는 실행하지 않습니다. 저장된
manifest가 이미 있으면 덮어쓰지 않습니다.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import yaml

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("프로젝트 루트(D:/ronbun)를 찾을 수 없습니다.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs/experiments/step2_pytorch_gradcam.yaml"
CONFIG = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))

MODEL_NAME = "arcface"     # "arcface", "adaface", "magface" 중 이번 실행 모델
MODE = "dev"               # 빠른 검증은 "dev", 전체 논문 실행만 "real"
DATA_FRACTION = 0.10       # identity 단위 사용 비율; 0 < 값 <= 1
SEED = 42                  # 부분집합·tie-break·random control 재현 seed
EXECUTE_STAGE = False      # 입력과 checkpoint를 채운 뒤 실제 계산할 때만 True
WRITE_OUTPUTS = False      # 검증 후 새 artifact를 저장할 때만 True

if MODEL_NAME not in CONFIG["models"]["selected"]:
    raise ValueError(f"지원하지 않는 모델: {MODEL_NAME}")
if MODE not in {"dev", "real"}:
    raise ValueError("MODE는 'dev' 또는 'real'이어야 합니다.")
if not 0.0 < DATA_FRACTION <= 1.0:
    raise ValueError("DATA_FRACTION은 (0, 1] 범위여야 합니다.")
if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True이면 EXECUTE_STAGE도 True여야 합니다.")

In [ ]:
from research.embeddings import (
    CheckpointProvenance,
    ModelSpec,
    PreprocessingSpec,
    write_model_spec,
)

CANDIDATE = CONFIG["models"]["candidates"][MODEL_NAME]

# checkpoint마다 반드시 실측·확인하여 채운다. 모델 이름만 보고
# 전처리나 target layer를 추측하지 않는다.
CHECKPOINT_PATH = None
CHECKPOINT_SOURCE_URL = None
IMPLEMENTATION_REPOSITORY = None
MODULE_FACTORY = None            # 예: "local_adapters.arcface:build_model"
TARGET_LAYER = None              # model.named_modules()의 정확한 경로
SOURCE_COLOR_ORDER = None        # 고정 crop artifact의 "rgb" 또는 "bgr"
MODEL_COLOR_ORDER = None         # checkpoint가 요구하는 "rgb" 또는 "bgr"
CHANNEL_MEAN = None              # 모델 채널 순서의 길이 3 tuple
CHANNEL_STD = None               # 모델 채널 순서의 길이 3 양수 tuple

In [ ]:
required = {
    "CHECKPOINT_PATH": CHECKPOINT_PATH,
    "CHECKPOINT_SOURCE_URL": CHECKPOINT_SOURCE_URL,
    "IMPLEMENTATION_REPOSITORY": IMPLEMENTATION_REPOSITORY,
    "MODULE_FACTORY": MODULE_FACTORY,
    "TARGET_LAYER": TARGET_LAYER,
    "SOURCE_COLOR_ORDER": SOURCE_COLOR_ORDER,
    "MODEL_COLOR_ORDER": MODEL_COLOR_ORDER,
    "CHANNEL_MEAN": CHANNEL_MEAN,
    "CHANNEL_STD": CHANNEL_STD,
}

if EXECUTE_STAGE:
    missing = [name for name, value in required.items() if value is None]
    if missing:
        raise RuntimeError(f"checkpoint 등록값이 비어 있습니다: {missing}")
    checkpoint = CheckpointProvenance.from_file(
        CHECKPOINT_PATH,
        source_url=CHECKPOINT_SOURCE_URL,
    )
    preprocessing = PreprocessingSpec(
        input_height=CANDIDATE["preprocessing"]["input_size"][0],
        input_width=CANDIDATE["preprocessing"]["input_size"][1],
        source_color_order=SOURCE_COLOR_ORDER,
        model_color_order=MODEL_COLOR_ORDER,
        channel_mean=tuple(CHANNEL_MEAN),
        channel_std=tuple(CHANNEL_STD),
    )
    spec = ModelSpec(
        family=MODEL_NAME,
        architecture=CANDIDATE["backbone"],
        training_dataset=CANDIDATE["training_dataset"],
        implementation_repository=IMPLEMENTATION_REPOSITORY,
        checkpoint=checkpoint,
        preprocessing=preprocessing,
        target_layer=TARGET_LAYER,
        embedding_dim=CANDIDATE["embedding_dim"],
        module_factory=MODULE_FACTORY,
    )
    registration = spec.to_manifest()
    if WRITE_OUTPUTS:
        destination = (
            PROJECT_ROOT
            / "runs/step2/model_registry"
            / f"{spec.model_uid}.json"
        )
        write_model_spec(destination, spec)
        registration["written_to"] = str(destination)
else:
    registration = {
        "status": "not_executed",
        "reason": "EXECUTE_STAGE=False",
        "model_name": MODEL_NAME,
    }
registration

다음 단계는 `01_preprocessing_and_model_smoke.ipynb`입니다. 새
checkpoint나 전처리 값으로 바꾸면 기존 manifest를 수정하지 말고 새
`model_uid`로 다시 등록합니다.